# how to finetune llm on instruction model datasets 

In [1]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

In [2]:
from huggingface_hub import notebook_login

In [3]:
notebook_login()

In [4]:
## download model

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config= config_8bit,
    trust_remote_code =True,
    device_map = "auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train[:300]")

In [6]:
dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [7]:
dataset[0]

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [8]:
example_text = [{"role": "assistant", "content": dataset[0]["chosen"]}]
example_text

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [9]:
tokenizer.apply_chat_template(example_text, tokenize = False)


"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

### Note
- we can see that the system message appers even though we dont need it

In [10]:
def apply_chat_temp(example):
    example["prompt"] = [{"role": "user", "content": example["prompt"]}]

    return example


new_dataset = dataset.map(apply_chat_temp)

new_dataset[0]

{'prompt': [{'role': 'user',
   'content': 'Oh, I just saw the best meme - have you seen it?'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [11]:
new_template = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""

In [12]:
def format_dataset(example):
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False
    )
    example["chosen"] = new_template.format(model_answer=example["chosen"])
    example["rejected"] = new_template.format(model_answer=example["rejected"])
    return example
formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

In [13]:
model_8bit

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Q

In [14]:
from peft import LoraConfig , get_peft_model

lora_config = LoraConfig(
    r=32,
    lora_alpha= 32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0.05,
    bias="none",
    task_type="CAUSAL_LM" 
    
)
lora_model = get_peft_model(model_8bit, lora_config)

lora_model.print_trainable_parameters()
 



trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [15]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [20]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir = "./results",
    per_device_train_batch_size = 2,
    ## in reality give epoch
    ##Max_steps
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    optim = "adamw_torch",
    warmup_steps = 5,
    logging_steps = 10,
    ## it uses a beta model as the reference and the value normally tells how determinent the output on the base model
    beta = 0.1,
    report_to = "none"
)

In [17]:
trainer = DPOTrainer(
    model = lora_model,
    # ref_model => u can point to a specific model fine tuned one , if left unset , it will use the non lora version of the current model
    train_dataset = formatted_dataset,
    processing_class = tokenizer,
    args=dpo_config
)

Dropping fully truncated examples from train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

In [18]:
import logging 
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

In [19]:
trainer.train()

Step,Training Loss
10,0.522155
20,0.020352
30,0.000062
40,0.000007
50,0.000002
60,0.000019
70,0.000012
80,0.000021
90,0.000003
100,0.000017


TrainOutput(global_step=114, training_loss=0.04760258736422928, metrics={'train_runtime': 365.6078, 'train_samples_per_second': 2.462, 'train_steps_per_second': 0.312, 'total_flos': 4050312220446720.0, 'train_loss': 0.04760258736422928, 'entropy': 1.1744674103600639, 'num_tokens': 342606.0, 'logits/chosen': -1.2646398604122255, 'logits/rejected': -1.946523293074543, 'mean_token_accuracy': 0.5991033571107047, 'rewards/chosen': 3.79202025277274, 'rewards/rejected': -14.110507488250732, 'rewards/accuracies': 1.0, 'rewards/margins': 17.902527604784286, 'logps/chosen': -297.10351998465404, 'logps/rejected': -448.5805206298828, 'epoch': 3.0})

In [21]:
## save the model
lora_path = "./lora_adapaters"
lora_model.save_pretrained(lora_path)

In [22]:
## download model

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

lora_path = "./lora_adapaters"

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code =True,
    device_map = "auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [23]:
from peft import PeftModel

final_model= PeftModel.from_pretrained(base_model, lora_path)
final_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [24]:
final_model = final_model.merge_and_unload()
final_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [25]:
final_model.push_to_hub("shankuriakose/Qwen2.5-1.5B-Instruct")
tokenizer.push_to_hub("shankuriakose/Qwen2.5-1.5B-Instruct")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/shankuriakose/Qwen2.5-1.5B-Instruct/commit/a403f8650aa3249513d137eacf54e40764682041', commit_message='Upload tokenizer', commit_description='', oid='a403f8650aa3249513d137eacf54e40764682041', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shankuriakose/Qwen2.5-1.5B-Instruct', endpoint='https://huggingface.co', repo_type='model', repo_id='shankuriakose/Qwen2.5-1.5B-Instruct'), pr_revision=None, pr_num=None)

In [28]:

import torch
from transformers import pipeline

model_id = final_model,

pipe = pipeline(
    "text-generation",
    model= base_model,
    # torch_dtype=torch.bfloat16,
    device_map="auto", 
    tokenizer = tokenizer
               )

messages = [
    {'role': 'system', 'content': 'you are funny'},
 {'role': 'user',
  'content': 'tell a joke about fat widow'}
]
outputs = pipe(
    messages,
    max_new_tokens=256,
    temperature=1, ## 1 means high creativity
    top_p= 0.9, # it will filter the the top contributors that reach upto 90%
    top_k = 10, # it selects the top number, overlap => on the intersection
    do_sample=True,
    repetition_penalty=1.2
)
print(outputs[0]["generated_text"][-1]["content"])

Why did the fat widow go to church?
To get bigger! (or more specifically, "to find her Lord in larger proportions")
